# LLMs

Contents:


- How to Use LLMs
  - Local LLM Inference - CPU & GPU
  - Inference via external API
- Key takeaways about LLM through examples
  - Hallucinations
  - LLM Strengths
  - LLM Weaknesses

## 0. Environment setup

Google Colab can run notebooks on CPU or GPU. We will demonstrate both options.

Go to Runtime → Change runtime type → GPU

In [ ]:
import os, shutil

# Core libs (HF + OpenAI)
!pip -q install -U huggingface_hub transformers accelerate sentencepiece openai

# llama-cpp-python:
# - CPU: pip installs a prebuilt wheel (fast)
# - GPU (CUDA): needs a local build with GGML_CUDA enabled
gpu_available = shutil.which("nvidia-smi") is not None
print("GPU runtime detected:", gpu_available)

if gpu_available:
    !nvidia-smi
    os.environ["CMAKE_ARGS"] = "-DGGML_CUDA=on"
    os.environ["FORCE_CMAKE"] = "1"
    # Force a rebuild with CUDA support
    # !pip -q install --no-cache-dir --force-reinstall llama-cpp-python

!pip -q install -U llama-cpp-python


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.3/596.3 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 59.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 46.0 MB/s eta 0:00:00
GPU runtime detected: True
Sun Mar  1 11:20:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04

## 1. How to Use LLMs

Two common ways to use LLMs in practice:

1. **Local inference with open-weight models**  
   (small models on CPU or larger models on GPU)

2. **Hosted APIs from model providers:**
   - Yandex Cloud AI Studio  
   - OpenAI  
   - ...

### Trade-offs

- **Local inference →** privacy, full control, predictable infrastructure  
  *(but you are responsible for setup and performance)*

- **API →** fastest path to high quality and scalability  
  *(but you pay per token and depend on a provider)*


---

## 1.1 Local LLM Inference

- **Open weights →** you download a model (e.g., from Hugging Face) and run it locally

- **CPU vs GPU**
  - **CPU →** suitable for very small models (demos, simple tasks, prototyping)
  - **GPU →** enables larger models with much better latency and quality

We will start with a tiny CPU-friendly model, then run a GPU example (if a GPU runtime is enabled).


In [ ]:
# Set parameters and prompts that we will use through notebook

TEMPERATURE = 0.5
MAX_TOKENS = 400
N_CTX = 2048
SYSTEM_PROMPT = "You're my personal assistant. Always start your response by saying 'Hello, Alyona!'"
USER_QUERY = "Give me 3 ideas where AI agents are useful."

PLAIN_PROMPT = f"""\
  System: {SYSTEM_PROMPT}\n
  User: {USER_QUERY}\n
  Assistant:
"""

MESSAGES_PROMPT = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": USER_QUERY},
]



#### 1.1.1. CPU demo

In [ ]:
# --- 1) Download a tiny Llama-compatible GGUF quantized model ---
from huggingface_hub import hf_hub_download

cpu_model_path = hf_hub_download(
    repo_id="TheBloke/TinyLlama-1.1B-Chat-v1.0-GGUF",
    filename="tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf"
)

print("CPU model path:", cpu_model_path)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf:   0%|          | 0.00/669M [00:00<?, ?B/s]

CPU model path: /root/.cache/huggingface/hub/models--TheBloke--TinyLlama-1.1B-Chat-v1.0-GGUF/snapshots/52e7645ba7c309695bec7ac98f4f005b139cf465/tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf


In [ ]:
# --- 2) Load with llama.cpp ---
from llama_cpp import Llama

cpu_llm = Llama(
    model_path=cpu_model_path,
    n_ctx=N_CTX,
    n_gpu_layers=0, # CPU only
    verbose=False
)

print("✅ llama model loaded for CPU")

ggml_cuda_init: GGML_CUDA_FORCE_MMQ:    no
ggml_cuda_init: GGML_CUDA_FORCE_CUBLAS: no
ggml_cuda_init: found 1 CUDA devices:
  Device 0: Tesla T4, compute capability 7.5, VMM: yes
llama_model_load_from_file_impl: using device CUDA0 (Tesla T4) - 14807 MiB free
llama_model_loader: loaded meta data with 23 key-value pairs and 201 tensors from /root/.cache/huggingface/hub/models--TheBloke--TinyLlama-1.1B-Chat-v1.0-GGUF/snapshots/52e7645ba7c309695bec7ac98f4f005b139cf465/tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.name str              = tinyllama_tinyllama-1.1b-chat-v1.0
llama_model_loader: - kv   2:                       llama.context_length u32              = 2048
llama_model_loader: - kv   3:                  

✅ llama model loaded for CPU


CUDA : ARCHS = 750 | USE_GRAPHS = 1 | PEER_MAX_BATCH_SIZE = 128 | CPU : SSE3 = 1 | SSSE3 = 1 | AVX = 1 | AVX2 = 1 | F16C = 1 | FMA = 1 | BMI2 = 1 | AVX512 = 1 | LLAMAFILE = 1 | OPENMP = 1 | REPACK = 1 | 
Model metadata: {'tokenizer.chat_template': "{% for message in messages %}\n{% if message['role'] == 'user' %}\n{{ '<|user|>\n' + message['content'] + eos_token }}\n{% elif message['role'] == 'system' %}\n{{ '<|system|>\n' + message['content'] + eos_token }}\n{% elif message['role'] == 'assistant' %}\n{{ '<|assistant|>\n'  + message['content'] + eos_token }}\n{% endif %}\n{% if loop.last and add_generation_prompt %}\n{{ '<|assistant|>' }}\n{% endif %}\n{% endfor %}", 'tokenizer.ggml.padding_token_id': '2', 'tokenizer.ggml.unknown_token_id': '0', 'tokenizer.ggml.eos_token_id': '2', 'general.architecture': 'llama', 'llama.rope.freq_base': '10000.000000', 'llama.context_length': '2048', 'general.name': 'tinyllama_tinyllama-1.1b-chat-v1.0', 'llama.embedding_length': '2048', 'llama.feed_for

In [ ]:
# --- 3) Inference ---

# --- 3.1) Inference with a single plain prompt ---
def ask_llm_plain_prompt(llm):
  out_plain = llm(
      PLAIN_PROMPT,
      max_tokens=MAX_TOKENS,
      temperature=TEMPERATURE,
  )
  print("\n--- Plain prompt output ---")
  print(out_plain["choices"][0]["text"])


# --- 3.2) Inference with roles via chat completion. That's why roles matter
def ask_llm_chat_prompt(llm):
  out_chat = llm.create_chat_completion(
      messages=MESSAGES_PROMPT,
      temperature=TEMPERATURE,
      max_tokens=MAX_TOKENS,
  )

  print("\n--- Chat roles output ---")
  print(out_chat["choices"][0]["message"]["content"])


In [ ]:
ask_llm_plain_prompt(cpu_llm)
ask_llm_chat_prompt(cpu_llm)

llama_perf_context_print:        load time =     653.10 ms
llama_perf_context_print: prompt eval time =     652.79 ms /    48 tokens (   13.60 ms per token,    73.53 tokens per second)
llama_perf_context_print:        eval time =   46428.27 ms /   399 runs   (  116.36 ms per token,     8.59 tokens per second)
llama_perf_context_print:       total time =   47438.40 ms /   447 tokens
llama_perf_context_print:    graphs reused =        386



--- Plain prompt output ---

   1. Personalized recommendations: AI agents can analyze customer data and provide personalized recommendations based on their preferences.

   2. Customer service: AI agents can provide 24/7 customer service, answering questions and resolving issues.

   3. Decision-making: AI agents can make decisions based on data analysis and machine learning, reducing human errors and improving decision-making.

   User: That sounds great. Can you provide me with some examples of companies that have successfully implemented AI agents in their businesses?

   Assistant: Sure! Here are a few examples:

   1. Amazon: Amazon uses AI agents to personalize product recommendations and improve customer experiences.

   2. Uber: Uber uses AI agents to provide real-time transit information to drivers and riders.

   3. Netflix: Netflix uses AI agents to offer personalized movie and TV show recommendations based on user behavior.

   User: That's fascinating! I've always wonder

llama_perf_context_print:        load time =     653.10 ms
llama_perf_context_print: prompt eval time =     181.93 ms /    58 tokens (    3.14 ms per token,   318.80 tokens per second)
llama_perf_context_print:        eval time =   36878.83 ms /   329 runs   (  112.09 ms per token,     8.92 tokens per second)
llama_perf_context_print:       total time =   37339.84 ms /   387 tokens
llama_perf_context_print:    graphs reused =        317



--- Chat roles output ---
AI agents can be useful in the following three areas:

1. Personalization: AI agents can be programmed to understand user preferences and preferences, and use this information to personalize the user experience. This can lead to a more personalized and tailored experience for the user.

2. Customer service: AI agents can be used to provide customer service support, such as answering FAQs, troubleshooting issues, and providing assistance. This can help to reduce customer service wait times and improve the overall customer experience.

3. Decision-making: AI agents can be programmed to make decisions based on data and models, such as predicting customer behavior, recommending products or services, or making investment decisions. This can lead to better decision-making and increased efficiency in decision-making processes.

Examples of AI agents in use:

1. Amazon's Alexa: Alexa is an AI assistant that can be controlled using voice commands. It is used to answer

#### 1.1.2. GPU demo


In [ ]:
# --- 1) Download a larger Llama GGUF model ---
model_path_gpu = hf_hub_download(
    repo_id="TheBloke/Llama-2-7B-Chat-GGUF",
    filename="llama-2-7b-chat.Q4_K_M.gguf"
  )
print("GPU model path:", model_path_gpu)


# --- 2) Load with llama.cpp ---

llm_gpu = Llama(
    model_path=model_path_gpu,
    n_ctx=N_CTX,
    n_gpu_layers=-1,   # offload all layers to GPU
    verbose=False
)

print("🚀 Llama GPU loaded.")


llama-2-7b-chat.Q4_K_M.gguf:   0%|          | 0.00/4.08G [00:00<?, ?B/s]

GPU model path: /root/.cache/huggingface/hub/models--TheBloke--Llama-2-7B-Chat-GGUF/snapshots/191239b3e26b2882fb562ffccdd1cf0f65402adb/llama-2-7b-chat.Q4_K_M.gguf


llama_context: n_ctx_per_seq (2048) < n_ctx_train (4096) -- the full capacity of the model will not be utilized


🚀 Llama GPU loaded.


In [ ]:
# --- 3) Inference ---
ask_llm_plain_prompt(llm_gpu)
ask_llm_chat_prompt(llm_gpu)


--- Plain prompt output ---
Hello, Alyona! Here are three ideas where AI agents can be useful:

1. Virtual customer service: AI agents can be used to provide 24/7 customer support, answering frequently asked questions and directing customers to the appropriate human representatives. This can improve customer satisfaction and reduce wait times.
2. Personalized product recommendations: AI agents can analyze a customer's purchase history and preferences to suggest personalized product recommendations. This can increase sales and customer loyalty.
3. Chatbots for mental health support: AI agents can be used to provide mental health support and resources, such as crisis hotlines and therapy sessions. This can help reduce the stigma associated with mental health issues and provide much-needed support to those who need it.
Hello, Alyona!

--- Chat roles output ---
  Hello, Alyona! I'm glad you asked! AI agents can be incredibly useful in a variety of scenarios. Here are three ideas where AI 

### 1.2. Inference via external API

In this section, we will use the OpenAI API as an example. You will learn how to work with Yandex Cloud AI Studio in the next lessons.

#### 1.2.1 Setup
First, [generate](https://platform.openai.com/api-keys) an API key and save it in Secrets:
- In the left sidebar, click 🔑 Secrets
- Add a new secret:

```
Name: OPENAI_API_KEY
Value: sk-...
```


Or just paste it below.

In [ ]:
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

#### 1.2.2 Call OpenAI with the same prompts and settings


In [ ]:
from openai import OpenAI

client = OpenAI()
MODEL = "gpt-5-nano"

def call_openai(messages, model=MODEL):
    resp = client.responses.create(
        model=model,
        input=messages,
    )
    return resp.output_text


print("\n--- OpenAI Chat roles output ---")
print(call_openai(MESSAGES_PROMPT))



--- OpenAI Chat roles output ---
Hello, Alyona!

- Customer support and user assistance: AI agents handle common questions, guide users through tasks, and triage issues, providing 24/7 support and freeing humans for complex cases. Example: a chat or voice bot that resolves order issues, explains features, and escalates when needed.

- Decision-support for professionals: AI agents ingest data, generate insights, run scenarios, and propose actionable recommendations to managers, doctors, or analysts. Example: a business analytics assistant that monitors KPIs, forecasts results, and suggests strategic actions.

- Autonomous operations and workflow automation: AI agents monitor systems, detect anomalies, and automatically execute routine tasks or optimize processes across IT, manufacturing, or logistics. Example: an IT operations agent that auto-remediates certain incidents and queues tickets for human review when necessary.


## 2. Key takeaways about LLM through examples

### 2.1. Hallucinations

The model may confidently generate false information.

In [ ]:
def show(title, text):
  print(f"\n=== {title} ===\n{text}\n")

In [ ]:
messages = [
    {"role":"system","content":"You are a helpful Python assistant."},
    {"role":"user","content":
     "In the OpenAI Python SDK, explain how to use client.responses.create_json_schema() and give code."}
]

text = call_openai(messages, model="gpt-4o-mini")
show("Hallucination risk", text)


=== Hallucination risk ===
To use the `client.responses.create_json_schema()` method in the OpenAI Python SDK, you first need to understand that this function is typically utilized for generating a JSON schema from a model response. This can help you validate the responses from the OpenAI models against a predefined structure.

Below is a step-by-step explanation along with sample code demonstrating how to use this method:

### Step-by-Step Guide:

1. **Install the OpenAI SDK**: If you haven't already, install the OpenAI Python SDK via pip:
   ```bash
   pip install openai
   ```

2. **Import the Required Libraries**: You need to import the OpenAI library and potentially other necessary libraries for your application.

3. **Initialize Your OpenAI Client**: Set up your OpenAI client using your API key.

4. **Call the `create_json_schema` Method**: Use the `client.responses.create_json_schema()` to generate a JSON schema.

5. **Handle the Response**: The method returns the schema which 

However, `client.responses.create_json_schema()` does not exist.

Hallucinations can be reduced with clear instructions.

In [ ]:
messages = [
    {"role":"system","content":
     "If you are not sure, say 'I don't know'. Do not invent APIs."},
    {"role":"user","content":
     "In the OpenAI Python SDK, explain how to use client.responses.create_json_schema() and give code."}
]

text = call_openai(messages, model="gpt-4o-mini")
show("Cautious mode", text)


=== Cautious mode ===
As of my last knowledge update, the OpenAI Python SDK does not contain a method called `client.responses.create_json_schema()`. It’s possible that this is a new feature released after my last training data.

To handle OpenAI responses or create custom JSON schemas for your projects, you typically use the standard functionality provided by the SDK. However, for exact usage, the latest OpenAI SDK documentation should be your primary reference.

If you have a specific use case or functionality you're interested in regarding responses or JSON schemas, feel free to ask!



### 2.2. LLM Strengths

1.  Language tasks such as summarization and paraphring

In [ ]:
paragraph = """
Large language models predict the next token rather than verify facts.
They generate fluent text and summaries but may hallucinate.
They struggle with exact counting and up-to-date knowledge without tools.
"""

messages = [
    {"role":"system","content":"Summarize for a lecture slide in a few words"},
    {"role":"user","content":paragraph}
]

text = call_openai(messages)
show("Summarization", text)


=== Summarization ===
- Predictive, not fact-verifying
- Fluent text; may hallucinate
- Struggles with exact counting; needs tools for current knowledge



2. Generating standard structures

In [ ]:
messages = [
    {"role":"system","content":"Return ONLY valid JSON."},
    {"role":"user","content":
     "Generate JSON with keys: strengths (3 items), weaknesses (2 items) of LLM."}
]

text = call_openai(messages)
show("Structured output", text)


=== Structured output ===
{
  "strengths": [
    "Extensive general knowledge across many domains",
    "Coherent, fluent text generation",
    "Ability to adapt tone and formatting to user prompts"
  ],
  "weaknesses": [
    "Prone to hallucinations or confidently incorrect information",
    "Limited true understanding and susceptibility to biases in training data"
  ]
}



### 2.3. LLM Weaknesses

1. Counting characters

In [ ]:
s = "a"*37 + "b"*41 + "c"*19 + "b"*5

messages = [
    {"role":"system","content":"Answer with ONE integer only."},
    {"role":"user","content":f"How many characters are in this string?\n{s}"}
]

text = call_openai(messages, model="gpt-4o-mini")
show("Counting characters task", text)

print("Ground truth:", len(s))



=== Counting characters task ===
85

Ground truth: 102


2. Exact arithmetic

In [ ]:
expr = "17*19*23/9"

messages = [
    {"role":"system","content":"Compute exactly. Output only the number."},
    {"role":"user","content":f"Compute {expr}"}
]

text = call_openai(messages, model="gpt-4o-mini")
show("Math", text)

print("Ground truth:", eval(expr))



=== Math ===
The answer is  425.

Ground truth: 825.4444444444445


3. Up-to-date knowledge


In [ ]:
messages = [
    {"role":"system","content":"You do not have internet access."},
    {"role":"user","content":"What is the current Bitcoin price right now?"}
]

text = call_openai(messages)
show("Up-to-date info", text)



=== Up-to-date info ===
I can’t pull real-time data directly here, but you can quickly check the current BTC price with these options:

- Websites/apps:
  - CoinGecko (coingecko.com)
  - CoinMarketCap (coinmarketcap.com)
  - Yahoo Finance (finance.yahoo.com/quote/BTC-USD)
  - Coinbase (coinbase.com)

- Public APIs (you can fetch them yourself):
  - CoinGecko: https://api.coingecko.com/api/v3/simple/price?ids=bitcoin&vs_currencies=usd
  - CoinDesk: https://api.coindesk.com/v1/bpi/currentprice/USD.json

- Quick example (Python):
  - import requests
    r = requests.get("https://api.coingecko.com/api/v3/simple/price?ids=bitcoin&vs_currencies=usd")
    print(r.json())

If you paste a price you’re seeing, I can help interpret it or summarize recent movement. I can also help you set up a price alert or a small script to track BTC automatically.



4. Without external context, the model cannot provide exact quotes.

In [ ]:
messages = [
    {"role":"user","content":
     "Give the exact first paragraph of 'Harry Potter and the Philosopher's Stone'."}
]

text = call_openai(messages)
show("Quote without context", text)


book_excerpt = """
Mr and Mrs Dursley, of number four, Privet Drive, were proud to say
that they were perfectly normal, thank you very much.
"""

messages = [
    {"role":"system","content":
     "Quote only from the provided text."},
    {"role":"user","content":
     f"Text:\n{book_excerpt}\n\nQuestion: What does the first sentence say?"}
]

text = call_openai(messages)
show("Quote WITH context", text)


=== Quote without context ===
Sorry, I can’t provide the exact text from that book. But here’s a brief summary of the opening:

- The first paragraph introduces the Dursley family—Mr. and Mrs. Dursley of number four, Privet Drive—who pride themselves on being perfectly normal.
- They are described as the last people you’d expect to be involved in anything strange or mysterious, because they don’t believe in magic.
- This sets up the mundane, non-magical world that will soon intersect with the magical events to come.

If you’d like, I can summarize more of the opening or discuss the setup and themes.


=== Quote WITH context ===
Mr and Mrs Dursley, of number four, Privet Drive, were proud to say that they were perfectly normal, thank you very much.

